In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# File paths
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/processed data set new.csv'
RESULTS_PATH = '/content/drive/MyDrive/Colab Notebooks/Results'

print("="*80)
print("XGBoost Job Title Classification with Hyperparameter Tuning")
print("="*80)

# Step 1: Load the dataset
print("\n[1/8] Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Step 2: Data exploration
print("\n[2/8] Data Exploration...")
print(f"\nTarget variable (job_title) distribution:")
print(df['job_title'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nData types:")
print(df.dtypes)

# Step 3: Prepare features and target
print("\n[3/8] Preparing features and target...")
X = df.drop('job_title', axis=1)
y = df['job_title']

# Encode categorical features if any
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    print(f"Encoding categorical columns: {categorical_cols}")
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

# Filter out classes with only one sample
class_counts = y.value_counts()
classes_to_keep = class_counts[class_counts > 1].index
df_filtered = df[y.isin(classes_to_keep)]

X_filtered = X.loc[df_filtered.index]
y_filtered = df_filtered['job_title']

# Encode target variable
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y_filtered)
print(f"Number of unique job titles (after filtering): {len(target_encoder.classes_)}")
print(f"Feature matrix shape (after filtering): {X_filtered.shape}")

# Step 4: Train-test split
print("\n[4/8] Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Step 5: Hyperparameter tuning with GridSearchCV
print("\n[5/8] Performing Hyperparameter Tuning with Cross-Validation...")
print("This may take several minutes...")

# Define parameter grid for tuning
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

# Initialize XGBoost classifier
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(target_encoder.classes_),
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist'
)

# Perform GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Step 6: Train final model with best parameters
print("\n[6/8] Training final model with best parameters...")
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

# Step 7: Cross-validation on best model
print("\n[7/8] Performing Cross-Validation on best model...")
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Step 8: Model evaluation
print("\n[8/8] Evaluating model on test set...")
y_pred = best_model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\nTest Set Results:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Detailed classification report
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

# Step 9: Save results
print("\n[9/9] Saving results...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save the trained model
model_filename = f"{RESULTS_PATH}/xgboost_model_{timestamp}.pkl"
with open(model_filename, 'wb') as f:
    pickle.dump(best_model, f)
print(f"Model saved: {model_filename}")

# Save label encoder
encoder_filename = f"{RESULTS_PATH}/label_encoder_{timestamp}.pkl"
with open(encoder_filename, 'wb') as f:
    pickle.dump(target_encoder, f)
print(f"Label encoder saved: {encoder_filename}")

# Save feature encoders if any
if categorical_cols:
    feature_encoder_filename = f"{RESULTS_PATH}/feature_encoders_{timestamp}.pkl"
    with open(feature_encoder_filename, 'wb') as f:
        pickle.dump(label_encoders, f)
    print(f"Feature encoders saved: {feature_encoder_filename}")

# Save results summary
results_summary = {
    'timestamp': timestamp,
    'best_parameters': grid_search.best_params_,
    'best_cv_score': float(grid_search.best_score_),
    'cv_scores': cv_scores.tolist(),
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'test_accuracy': float(accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1_score': float(f1),
    'num_classes': len(target_encoder.classes_),
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'feature_count': int(X_filtered.shape[1]) # Use X_filtered shape for feature count
}

results_filename = f"{RESULTS_PATH}/training_results_{timestamp}.json"
with open(results_filename, 'w') as f:
    json.dump(results_summary, f, indent=4)
print(f"Results summary saved: {results_filename}")

# Feature importance visualization
print("\nGenerating feature importance plot...")
feature_importance = pd.DataFrame({
    'feature': X_filtered.columns, # Use X_filtered columns
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][:20], feature_importance['importance'][:20])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
importance_plot = f"{RESULTS_PATH}/feature_importance_{timestamp}.png"
plt.savefig(importance_plot, dpi=300, bbox_inches='tight')
print(f"Feature importance plot saved: {importance_plot}")
plt.close()

# Confusion matrix visualization
print("Generating confusion matrix...")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_encoder.classes_,
            yticklabels=target_encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
cm_plot = f"{RESULTS_PATH}/confusion_matrix_{timestamp}.png"
plt.savefig(cm_plot, dpi=300, bbox_inches='tight')
print(f"Confusion matrix saved: {cm_plot}")
plt.close()

# CV scores visualization
print("Generating cross-validation scores plot...")
plt.figure(figsize=(8, 6))
plt.plot(range(1, len(cv_scores) + 1), cv_scores, marker='o', linestyle='-', linewidth=2, markersize=8)
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Cross-Validation Scores')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
cv_plot = f"{RESULTS_PATH}/cv_scores_{timestamp}.png"
plt.savefig(cv_plot, dpi=300, bbox_inches='tight')
print(f"Cross-validation scores plot saved: {cv_plot}")
plt.close()

print("\n" + "="*80)
print("Training Complete!")
print("="*80)
print(f"\nAll results saved to: {RESULTS_PATH}")
print(f"Model file: xgboost_model_{timestamp}.pkl")
print(f"Results summary: training_results_{timestamp}.json")
print("\nYou can load the model later using:")
print("with open('model_file.pkl', 'rb') as f:")
print("    model = pickle.load(f)")

XGBoost Job Title Classification with Hyperparameter Tuning

[1/8] Loading dataset...
Dataset shape: (4793, 66)
Columns: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_title_clean', 'company_clean', 'job_skills_clean', 'job_location_clean', 'job_title_word_count', 'skills_word_count', 'job_title_char_count', 'skills_char_count', 'skills_list_count', 'has_skill_communication', 'has_skill_customer_service', 'has_skill_teamwork', 'has_skill_leadership', 'has_skill_communication_skills', 'job_title_tokens', 'job_skills_tokens', 'title_tokens_count', 'skills_tokens_count', 'combined_text_clean', 'job_level_encoded', 'job_type_onsite', 'search_country_encoded', 'job_title_word_count_scaled', 'skills_word_count_scaled', 'job_title_char_count_scaled', 'skills_char_count_scaled', 'skills_list_count_scaled', 'title

ValueError: The test_size = 225 should be greater or equal to the number of classes = 311